In [1]:
!pip install Starfysh

In [3]:
import os
import numpy as np
import pandas as pd
import torch

In [5]:
!pip install histomicstk --no-deps
!pip show histomicstk | grep "Requires"

In [ ]:
!pip install large-image!pip install large-image

In [ ]:
!pip install ctk-cli

In [ ]:
!pip install distributed

In [ ]:
!pip install girder-client

In [ ]:
!pip install girder-slicer-cli-web

In [ ]:
pip install pandas==1.5.3

In [ ]:
!pip install shapely

In [ ]:
from starfysh import (AA, utils, plot_utils, post_analysis)
from starfysh import starfysh as sf_model

In [ ]:
!pip install scanpy

In [ ]:
!pip install scikit-image

In [ ]:
path = 

In [ ]:
sig_name = f'{path}/bc_signatures_version_1013.csv'

In [ ]:
adata, adata_normed = utils.load_adata(data_folder=data_path,
                                       sample_id=sample_id,
                                       n_genes=2000)

gene_sig = pd.read_csv(os.path.join(data_path, sig_name))
gene_sig = utils.filter_gene_sig(gene_sig, adata.to_df())
gene_sig.head()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(2, 2), dpi=300)
plt.imshow(img)
plt.show()

In [ ]:
print("Shape of adata:", adata.shape)
print("Shape of adata_normed:", adata_normed.shape)
print("Shape of gene_sig:", gene_sig.shape)

In [ ]:
visium_args = utils.VisiumArguments(adata,
                                    adata_normed,
                                    gene_sig,
                                    img_metadata,
                                    window_size=3, 
                                    sample_id=sample_id)
adata, adata_normed = visium_args.get_adata()

In [ ]:
aa_model = AA.ArchetypalAnalysis(adata_orig=adata_normed)
archetype, arche_dict, major_idx, evs = aa_model.compute_archetypes()

arche_df = aa_model.find_archetypal_spots(major=True)

markers_df = aa_model.find_markers(display=False)

anchors_df = visium_args.get_anchors()
anchor_percent = 0.05
n_top_anchors = int(anchor_percent*adata.shape[0])
map_df, map_dict = aa_model.assign_archetypes(anchor_df=anchors_df[:n_top_anchors],
                                              r=n_top_anchors)

In [ ]:
import seaborn as sns

In [ ]:
map_df_prop = map_df / n_top_anchors
fig, ax = plt.subplots(figsize=(5, 5), dpi=300)
g = sns.heatmap(map_df_prop[list(map_dict.values())], cmap='Purples', vmax=0.5, square=True,
                cbar_kws=dict(location='left'), ax=ax)
ax.yaxis.tick_right()
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.show()

In [ ]:
visium_args = utils.VisiumArguments(adata,
                                    adata_normed,
                                    gene_sig,
                                    img_metadata,
                                    n_anchors=60,
                                    window_size=3,
                                    sample_id=sample_id
                                   )

adata, adata_normed = visium_args.get_adata()
anchors_df = visium_args.get_anchors()

In [ ]:
img_metadata = utils.preprocess_img(data_path,
                                    sample_id,
                                    adata_index=adata.obs.index,
                                    hchannel=False
                                    )
img, map_info, scalefactor = img_metadata['img'], img_metadata['map_info'], img_metadata['scalefactor']
umap_df = utils.get_umap(adata, display=True)

In [ ]:
plot_utils.plot_anchor_spots(umap_df,
                             visium_args.pure_spots,
                             visium_args.sig_mean,
                             bbox_x=2
                            )

In [ ]:
aa_model.plot_mapping(map_df)

In [ ]:
visium_args = utils.refine_anchors(
    visium_args,
    aa_model,
    thld=0.7,  # alignment threshold
    n_genes=5,
    n_iters=1
)

# Get updated adata & signatures
adata, adata_normed = visium_args.get_adata()
gene_sig = visium_args.gene_sig
cell_types = gene_sig.columns

In [ ]:
n_repeats = 3
epochs = 200
patience = 50
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model, loss = utils.run_starfysh(visium_args,
                                 n_repeats=n_repeats,
                                 epochs=epochs,
                                 poe=False
                                )

In [ ]:
plot_utils.pl_spatial_inf_feature(adata,
                                  feature='qc_m',
                                  vmax=0.1,
                                  spot_size=300)

In [ ]:
plot_utils.pl_spatial_inf_feature(adata,
                                  feature='qz_m',
                                  factor=['Basal', 'LumA', 'MBC', 'Normal epithelial'],
                                  spot_size=3,
                                  vmax=0.2)

In [ ]:
pred_exprs = sf_model.model_ct_exp(model,
                                   adata,
                                   visium_args,
                                   device=device)

In [ ]:
outdir = ''
if not os.path.exists(outdir):
    os.mkdir(outdir)
torch.save(model.state_dict(), os.path.join(outdir, 'starfysh_model.pt'))
adata.write(os.path.join(outdir, 'st.h5ad'))

In [ ]:
cell_type_labels = adata.uns['cell_types']
qc_m_df = pd.DataFrame(adata.obsm['qc_m'], index=adata.obs.index, columns=cell_type_labels)
adata.obs['most_likely_cell_type'] = qc_m_df.idxmax(axis=1)